# 03 — Integração e limpeza

**Objetivo:** juntar em uma única base, por município de SP:
- idosos sozinhos / domicílios com responsável idoso (notebook 01)
- internações por causa e ano (notebook 02)
- IDH municipal (controle)

**Como as bases são cruzadas:** por **nome de município normalizado**
(`config.normalizar_municipio`), não por código IBGE/DATASUS — nenhuma das
fontes reais que conseguimos traz os dois códigos ao mesmo tempo, e não
temos acesso à internet neste ambiente para baixar a lista oficial de
códigos do IBGE. O cruzamento por nome foi validado (ver notebook 02).

**Três saídas desta etapa, com propósitos diferentes:**
1. `dataset_consolidado_sp.csv` — **painel** (uma linha por município × ano,
   3.225 linhas, sem filtro de limpeza). Bom para totais e gráficos de
   evolução (o total de internações é real, não depende da qualidade da
   taxa) e para o estudo de caso de Rio Claro (notebook 05).
2. `dataset_municipios_sp_bruto.csv` — **nível município, sem limpeza**
   (645 linhas). Guardado para transparência/reprodutibilidade.
3. `dataset_municipios_sp.csv` — **nível município, limpo** (outliers e
   municípios sem IDH removidos — ver seção 3.5). **É esta que usamos para
   testar a hipótese principal no notebook 04.**

⚠️ **Por que existe um painel e uma base por município:** `pct_idosos_sozinhos`
vem do Censo 2022 — é o **mesmo valor nos 5 anos** de cada município. Se você
rodar a correlação/regressão direto no painel (3.225 linhas), cada município
entra 5 vezes com a mesma variável independente — isso é
[pseudorreplicação](https://en.wikipedia.org/wiki/Pseudoreplication): infla
artificialmente o "n" e faz resultados parecerem mais significativos do que
são. A tabela por município (uma observação independente por município) é a
forma estatisticamente correta de testar a hipótese.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd


## 3.1 Carregar as bases já processadas


In [ ]:
censo = pd.read_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv")
internacoes = pd.read_csv(config.DATA_PROCESSED / "internacoes_sp.csv")

print("censo:", censo.shape, "-> municípios:", censo["municipio_norm"].nunique())
print("internacoes:", internacoes.shape, "-> municípios:", internacoes["municipio_norm"].nunique())


## 3.2 IDH municipal (controle)

`data/external/idh_sp.csv` (colunas `municipio`, `idh`) — ver
`data/external/FONTES_RIO_CLARO.md` para a origem exata dos valores.

⚠️ **Cobertura parcial: 259 dos 645 municípios de SP têm IDH nesse arquivo**
(uma 260ª linha, "Guaxupé", nem é de SP — é de MG, provavelmente um erro no
arquivo de origem, e cai fora do cruzamento sem problema). Os municípios
sem IDH entram como `NaN` aqui — a decisão sobre o que fazer com eles
(eliminar, já que não há como imputar com confiança) é tomada na seção 3.5.

Essa variável entra como **controle**: o objetivo é checar se a associação
entre "idosos sozinhos" e "internações" se mantém mesmo depois de levar em
conta a renda/desenvolvimento do município.


In [ ]:
caminho_idh = config.DATA_EXTERNAL / "idh_sp.csv"

idh = pd.read_csv(caminho_idh).rename(columns={"idh": "idhm"})
idh["municipio_norm"] = idh["municipio"].apply(config.normalizar_municipio)
idh = idh[["municipio_norm", "idhm"]]

print(f"{len(idh)} municípios com IDH no arquivo (de 645 no estado)")
idh.head()


## 3.3 Montar o painel (município × ano)

O merge parte da lista **completa** de municípios (os 645 do Censo,
cruzados com todos os anos do SIH) e preenche com 0 onde não há registro de
internação — em vez de partir de `internacoes`, o que faria os municípios
sem nenhuma internação registrada desaparecerem da base em vez de entrar
com 0.


In [ ]:
internacoes_wide = (
    internacoes
    .pivot_table(index=["municipio_norm", "ano"], columns="causa", values="internacoes", fill_value=0)
    .reset_index()
)
causas_cols = list(config.CAUSAS_SIH.keys())

anos_sih = sorted(internacoes["ano"].unique())
grid = censo[["municipio", "municipio_norm"]].merge(pd.DataFrame({"ano": anos_sih}), how="cross")

painel = grid.merge(internacoes_wide, on=["municipio_norm", "ano"], how="left")
painel[causas_cols] = painel[causas_cols].fillna(0)
painel["total_internacoes_causas_estudo"] = painel[causas_cols].sum(axis=1)

painel = painel.merge(censo, on=["municipio", "municipio_norm"], how="left")
painel = painel.merge(idh, on="municipio_norm", how="left")

# Taxa por 100 mil DOMICÍLIOS com responsável idoso (não por 100 mil idosos-pessoa --
# ver a nota metodológica do notebook 01 sobre a diferença). Arredondada a 2 casas --
# a precisão extra não tem significado real e só atrapalha a leitura das tabelas.
painel["taxa_internacao_100k_domicilios_idosos"] = (
    painel["total_internacoes_causas_estudo"] / painel["domicilios_resp_idoso"] * 100_000
).round(2)

print(painel.shape, "(esperado 645 x 5 = 3225)")
print("linhas com IDH:", painel["idhm"].notna().sum())
painel.head()


In [ ]:
painel.to_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv", index=False)
print("Salvo em", config.DATA_PROCESSED / "dataset_consolidado_sp.csv")


## 3.4 Montar a base por município (nível de análise da hipótese principal)

Uma linha por município, com as internações **somadas** entre 2022 e 2026
(nota: 2026 é parcial, até julho) e as variáveis que não mudam por ano
(`pct_idosos_sozinhos`, `idhm`) entrando uma única vez.


In [ ]:
municipios_agg = painel.groupby(["municipio", "municipio_norm"]).agg(
    total_internacoes=("total_internacoes_causas_estudo", "sum"),
    domicilios_resp_idoso=("domicilios_resp_idoso", "first"),
    idosos_sozinhos=("idosos_sozinhos", "first"),
    pct_idosos_sozinhos=("pct_idosos_sozinhos", "first"),
    idhm=("idhm", "first"),
).reset_index()

municipios_agg["taxa_internacao_100k_domicilios_idosos"] = (
    municipios_agg["total_internacoes"] / municipios_agg["domicilios_resp_idoso"] * 100_000
).round(2)

print(municipios_agg.shape, "(esperado 645)")
municipios_agg.head()


## 3.5 Limpeza: outliers e campos incompletos

Três problemas de qualidade de dado nesta base, e a decisão tomada para
cada um (eliminar vs. arredondar/imputar):

**1. Casas decimais sem significado real** — já arredondado (seções 3.3/3.4)
para 2 casas. Arredondar aqui não perde informação: a 3ª casa decimal de
uma taxa por 100 mil não muda nenhuma decisão.

**2. Outliers na taxa de internação** — detectados pela regra de Tukey
(acima de Q3 + 1,5×IQR). **Decisão: eliminar, não arredondar/capar.** O
motivo não é só estatístico: os municípios que mais aparecem como outlier
(ver célula abaixo) são cidades-polo regionais em saúde (Presidente
Prudente, Botucatu, Jaú, Catanduva, Franco da Rocha...) — o que bate com a
suspeita já registrada em `FONTES_RIO_CLARO.md`: o SIH que temos é **"por
local de internação"** (onde fica o hospital), não **"por local de
residência"** (onde mora o paciente). Uma cidade com hospital regional
"recebe" no papel as internações de pacientes de outras cidades, inflando
sua taxa artificialmente. Isso não é ruído aleatório que dê pra suavizar
com um limite (cap) — é viés sistemático de definição geográfica. A forma
correta de resolver de verdade é reexportar o SIH "por local de
residência" (ver notebook 02); até lá, excluir esses municípios da análise
principal é a opção mais honesta.

**3. IDH ausente (~60% dos municípios)** — **decisão: eliminar as linhas
sem IDH da base de análise**, não imputar. Imputar (por exemplo com a
mediana) em 60% das linhas distorceria a regressão em vez de corrigi-la —
não temos uma fonte confiável de IDH por município vizinho ou mesorregião
disponível aqui (sem internet) para imputar com algum critério melhor que
"chutar a mediana para a maioria dos dados".


In [ ]:
# --- Outliers (regra de Tukey) ---
q1, q3 = municipios_agg["taxa_internacao_100k_domicilios_idosos"].quantile([0.25, 0.75])
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

municipios_agg["outlier_taxa"] = municipios_agg["taxa_internacao_100k_domicilios_idosos"] > limite_superior

print(f"Limite superior (Tukey, 1.5x IQR acima de Q3): {limite_superior:.0f}")
print(f"Outliers detectados: {municipios_agg['outlier_taxa'].sum()} de {len(municipios_agg)}")
municipios_agg[municipios_agg["outlier_taxa"]].sort_values(
    "taxa_internacao_100k_domicilios_idosos", ascending=False
)[["municipio", "domicilios_resp_idoso", "total_internacoes", "taxa_internacao_100k_domicilios_idosos"]]


In [ ]:
# --- Guarda a versão bruta (sem filtro) para transparência/reprodutibilidade ---
municipios_agg.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv", index=False)
print("Bruto salvo em", config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv")

# --- Versão limpa: remove outliers e municípios sem IDH -- é essa que o notebook 04 usa ---
municipios_limpo = municipios_agg[
    ~municipios_agg["outlier_taxa"] & municipios_agg["idhm"].notna()
].drop(columns="outlier_taxa").reset_index(drop=True)

print(f"\nBase limpa: {len(municipios_limpo)} municípios (de 645)")
print(f"  - removidos por outlier: {(municipios_agg['outlier_taxa'] & municipios_agg['idhm'].notna()).sum()}")
print(f"  - removidos por IDH ausente: {(~municipios_agg['outlier_taxa'] & municipios_agg['idhm'].isna()).sum()}")
print(f"  - removidos pelos dois motivos: {(municipios_agg['outlier_taxa'] & municipios_agg['idhm'].isna()).sum()}")

municipios_limpo.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv", index=False)
print("\nLimpo salvo em", config.DATA_PROCESSED / "dataset_municipios_sp.csv")


## 3.6 Destacar Rio Claro


In [ ]:
print("--- Painel (por ano) ---")
display(painel[painel["municipio"] == config.RIO_CLARO_NOME])

print("--- Agregado, base limpa (nível município) ---")
display(municipios_limpo[municipios_limpo["municipio"] == config.RIO_CLARO_NOME])
